# Smart MCQ Solver — Milestone 2
**Email:** 23f3001763@ds.study.iitm.ac.in



In [23]:

!pip install -q datasets sentence-transformers transformers scikit-learn pandas torch

In [24]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Load dataset ──────────────────────────────────────────────────────────────
file_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'

# Fallback for local run
if not os.path.exists(file_path):
    file_path = 'train.csv'

dataset = load_dataset('csv', data_files=file_path)['train']
train_df = pd.read_csv(file_path)
print("Dataset loaded successfully!")

Dataset loaded successfully!



## Q1 — Hugging Face datasets


In [25]:
# ── Q1 ────────────────────────────────────────────────────────────────────────
def combine(example):
    example['combined_text'] = str(example['prompt']) + " " + str(example['A'])
    return example

dataset_mapped = dataset.map(combine)
q1_ans = len(dataset_mapped[51]['combined_text'])
print(f">>> Q1 Answer (Length of combined_text at index 51) = {q1_ans}")

>>> Q1 Answer (Length of combined_text at index 51) = 614



## Q2 & Q3 — bert-base-uncased tokenizer


In [26]:
# ── Q2 & Q3 ───────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
q2_ans = tokenizer.vocab_size
q3_ans = tokenizer.sep_token_id

print(f">>> Q2 Answer (Vocabulary size) = {q2_ans}")
print(f">>> Q3 Answer ([SEP] token ID) = {q3_ans}")

>>> Q2 Answer (Vocabulary size) = 30522
>>> Q3 Answer ([SEP] token ID) = 102



## Q4 — Tokenization shape


In [27]:
# ── Q4 ────────────────────────────────────────────────────────────────────────
prompts_q4 = [str(p) if p is not None else "" for p in dataset['prompt']]
tokens = tokenizer(prompts_q4, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
q4_ans = tuple(tokens['input_ids'].shape)

print(f">>> Q4 Answer (Shape of input_ids tensor) = {q4_ans}")

>>> Q4 Answer (Shape of input_ids tensor) = (2000, 128)


---
## Q5 — Attention head dimensionality


In [28]:
# ── Q5 ────────────────────────────────────────────────────────────────────────
q5_ans = 768 // 12
print(f">>> Q5 Answer (Dimensionality of each attention head) = {q5_ans}")

>>> Q5 Answer (Dimensionality of each attention head) = 64



## Q6 & Q7 — BERT hidden states


In [29]:
# ── Q6 & Q7 ───────────────────────────────────────────────────────────────────
model = AutoModel.from_pretrained('bert-base-uncased')
prompt_0 = str(dataset[0]['prompt'])
tokens_0 = tokenizer(prompt_0, return_tensors='pt')

with torch.no_grad():
    outputs = model(**tokens_0)

q6_ans = tuple(outputs.last_hidden_state.shape)
cls_vector = outputs.last_hidden_state[0, 0, :]
q7_ans = round(float(torch.sum(cls_vector[:5])), 4)

print(f">>> Q6 Answer (Shape of last_hidden_state) = {q6_ans}")
print(f">>> Q7 Answer (Sum of first 5 [CLS] float values) = {q7_ans}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> Q6 Answer (Shape of last_hidden_state) = (1, 31, 768)
>>> Q7 Answer (Sum of first 5 [CLS] float values) = -1.2001



## Q8 — Attention weights


In [30]:
# ── Q8 ────────────────────────────────────────────────────────────────────────
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text_q8 = "Light-ion fusion is a technique."
tokens_attn = tokenizer(text_q8, return_tensors='pt')

input_ids = tokens_attn['input_ids'][0]
tokens_list = tokenizer.convert_ids_to_tokens(input_ids)
fusion_idx = tokens_list.index("fusion")

with torch.no_grad():
    outputs_attn = model_attn(**tokens_attn)

last_layer_attn = outputs_attn.attentions[-1]
q8_ans = round(last_layer_attn[0, 0, 0, fusion_idx].item(), 4)

print(f">>> Q8 Answer (Attention weight of [CLS] to fusion) = {q8_ans}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> Q8 Answer (Attention weight of [CLS] to fusion) = 0.1025



## Q9 — Sentence-Transformers


In [31]:
# ── Q9 ────────────────────────────────────────────────────────────────────────
st_model = SentenceTransformer('all-MiniLM-L6-v2')
opt_b_0 = str(dataset[0]['B'])

emb_prompt = st_model.encode(prompt_0)
emb_b = st_model.encode(opt_b_0)

q9_ans = round(util.cos_sim(emb_prompt, emb_b)[0][0].item(), 4)
print(f">>> Q9 Answer (Cosine similarity score) = {q9_ans}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> Q9 Answer (Cosine similarity score) = 0.7658



## Q10 — Pipeline comparison


In [32]:
# ── Q10 ───────────────────────────────────────────────────────────────────────
OPTIONS = ['A', 'B', 'C', 'D', 'E']

# Pipeline 1: TF-IDF
combined_docs = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
).tolist()
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_docs)

prompt_vecs = tfidf.transform(train_df['prompt'].fillna(''))
opt_vecs_tfidf = {opt: tfidf.transform(train_df[opt].fillna('')) for opt in OPTIONS}

tfidf_sims = np.column_stack([
    cosine_similarity(prompt_vecs, opt_vecs_tfidf[opt]).diagonal()
    for opt in OPTIONS
])
tfidf_top3_idx = np.argsort(-tfidf_sims, axis=1)[:, :3]
tfidf_preds = [[OPTIONS[i] for i in row] for row in tfidf_top3_idx]

# Pipeline 2: MiniLM
prompts_all = train_df['prompt'].fillna('').tolist()
opts_all = {opt: train_df[opt].fillna('').tolist() for opt in OPTIONS}

emb_prompts = st_model.encode(prompts_all, batch_size=64, convert_to_tensor=True)
emb_opts = {opt: st_model.encode(opts_all[opt], batch_size=64, convert_to_tensor=True) for opt in OPTIONS}

minilm_sims = np.column_stack([
    util.cos_sim(emb_prompts, emb_opts[opt]).diagonal().cpu().numpy()
    for opt in OPTIONS
])
minilm_top3_idx = np.argsort(-minilm_sims, axis=1)[:, :3]
minilm_preds = [[OPTIONS[i] for i in row] for row in minilm_top3_idx]

def mapk(actuals, predictions, k=3):
    scores = []
    for a, p in zip(actuals, predictions):
        score = 0.0
        for rank, pred in enumerate(p[:k], start=1):
            if pred == a:
                score = 1.0 / rank
                break
        scores.append(score)
    return np.mean(scores)

actuals = train_df['answer'].tolist()
q10_1_ans = round(mapk(actuals, minilm_preds), 4)

count_diff = 0
for a, p_tfidf, p_minilm in zip(actuals, tfidf_preds, minilm_preds):
    if a not in p_tfidf and a in p_minilm:
        count_diff += 1

print(f">>> Q10 Answer (MiniLM MAP@3) = {q10_1_ans}")
print(f">>> Q10 Answer (Count of diff) = {count_diff}")

>>> Q10 Answer (MiniLM MAP@3) = 0.4231
>>> Q10 Answer (Count of diff) = 531


## Q11 & Q12 — Zero-shot classification


In [33]:
# ── Q11 & Q12 ─────────────────────────────────────────────────────────────────
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
prompt_1 = str(dataset[1]['prompt'])
labels = [str(dataset[1]['A']), str(dataset[1]['B']), str(dataset[1]['C'])]

res = classifier(prompt_1, candidate_labels=labels)
q11_ans = round(res['scores'][0], 4)

res_multi = classifier(prompt_1, candidate_labels=labels, multi_label=True)
sum_prev = sum(res['scores'])
sum_multi = sum(res_multi['scores'])
q12_ans = round(abs(sum_prev - sum_multi), 4)

print(f">>> Q11 Answer (Top probability score) = {q11_ans}")
print(f">>> Q12 Answer (Absolute difference in sums) = {q12_ans}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

>>> Q11 Answer (Top probability score) = 0.4575
>>> Q12 Answer (Absolute difference in sums) = 0.9995



## Q13 — text2text-generation


In [35]:
# ── Q13 ───────────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

A_0 = str(dataset[0]['A'])
B_0 = str(dataset[0]['B'])
input_str = f"Question: {prompt_0}. Is the correct answer A: {A_0} or B: {B_0}? Answer with just the letter A or B."

input_ids = t5_tokenizer(input_str, return_tensors="pt").input_ids
outputs = t5_model.generate(input_ids, max_new_tokens=5)
q13_ans = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f">>> Q13 Answer (Exact string output) = {q13_ans}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

>>> Q13 Answer (Exact string output) = B
